In [23]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [24]:
# List of paths with .csv files
folder_names  = [
        "/mnt/c/users/helen/Desktop/U2OS_WT",
        "/mnt/c/users/helen/Desktop/U2OS_As_2h",
        "/mnt/c/users/helen/Desktop/U2OS_As_4h",
]

folders = [Path(f) for f in folder_names]

In [ ]:
# Data loading into single dataframe
dfs = []

for folder in folders:
    for file in folder.glob("*.csv"):

        df = pd.read_csv(file)

        # Add file name column
        df["file_name"] = file.stem

        dfs.append(df)

    # Combine all dataframes
    data = pd.concat(dfs, ignore_index=True)

    # Column ordering
    data = data[
        ["file_name", "ROI", "Area", "Mean"]
    ]

    # Rename ROI column to remove the suffix after the underscore
    data["ROI"] = data["ROI"].apply(lambda x: x.split("_")[0])
    
# Add new sample_name column
data['sample_name'] = data['file_name'].apply(lambda x: 'WT' if 'NO_AS' in x else ('Sodium_arsenite_2h' if '2h' in x else 'Sodium_arsenite_4h'))

In [ ]:
# Optional: Save the combined dataframe to a new CSV file
data.to_csv("/mnt/c/users/helen/Desktop/combined_data.csv", index=False, sep=";", decimal=",")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Boxplot
data.boxplot(
    column="Mean",
    by="ROI",
    ax=ax,
    grid=False,
    showfliers=False,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", alpha=0.5),
    medianprops=dict(linewidth=2),
)

# Add individual points (jitter)
groups = data.groupby("ROI")

for i, (_, group) in enumerate(groups, start=1):
    x = np.random.normal(i, 0.06, size=len(group))
    ax.scatter(
        x,
        group["Mean"],
        alpha=0.7,
        s=40,
    )

ax.set_title("Mean Intensity")
ax.set_xlabel("ROI")
ax.set_ylabel("Mean Intensity (a.u.)")

# Remove Pandas automatic title
plt.suptitle("")

# Cleaner appearance
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()